In [ ]:
!pip install scikit-learn pandas matplotlib seaborn transformers datasets torch accelerate evaluate

In [ ]:
# PART A: SVM + TF-IDF baseline
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_score
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC
from sklearn.metrics import classification_report, confusion_matrix, precision_recall_fscore_support, roc_auc_score, accuracy_score
import joblib
import seaborn as sns
import matplotlib.pyplot as plt

# 1) Load df (if not already in memory)
df = pd.read_csv("scam_dataset_preprocessed.csv")

# 2) Prepare X and y
X_text = df['final_text_for_traditional_ml'].astype(str).tolist()
y = df['label'].astype(int).values

# 3) Train/Val/Test split (stratified)
X_temp, X_test, y_temp, y_test = train_test_split(X_text, y, test_size=0.15, stratify=y, random_state=42)
X_train, X_val, y_train, y_val = train_test_split(X_temp, y_temp, test_size=0.17647, stratify=y_temp, random_state=42)
# above produces 70/15/15 approx

print("Sizes:", len(X_train), len(X_val), len(X_test))

# 4) Vectorize with TF-IDF
tfidf = TfidfVectorizer(max_features=5000, ngram_range=(1,2))  # tune max_features if needed
X_train_tfidf = tfidf.fit_transform(X_train)
X_val_tfidf   = tfidf.transform(X_val)
X_test_tfidf  = tfidf.transform(X_test)

# 5) Train SVM with class_weight balanced
svm = LinearSVC(class_weight='balanced', max_iter=10000, random_state=42)
# Quick grid search for C
params = {'C':[0.01, 0.1, 1, 10]}
gs = GridSearchCV(svm, params, cv=3, scoring='f1', n_jobs=-1, verbose=1)
gs.fit(X_train_tfidf, y_train)
print("Best params:", gs.best_params_)
best_svm = gs.best_estimator_

# 6) Evaluate on validation and test
def eval_model(model, X_vec, y_true, label=''):
    y_pred = model.predict(X_vec)
    print(f"\n=== Evaluation: {label} ===")
    print(classification_report(y_true, y_pred, digits=4))
    cm = confusion_matrix(y_true, y_pred)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
    plt.title(f"Confusion Matrix - {label}")
    plt.xlabel("Predicted")
    plt.ylabel("Actual")
    plt.show()
    return y_pred

y_val_pred = eval_model(best_svm, X_val_tfidf, y_val, label='Validation (SVM)')
y_test_pred = eval_model(best_svm, X_test_tfidf, y_test, label='Test (SVM)')

# 7) Save TF-IDF and model for later
joblib.dump(tfidf, "tfidf_vectorizer.joblib")
joblib.dump(best_svm, "svm_tfidf_model.joblib")
print("Saved tfidf_vectorizer.joblib and svm_tfidf_model.joblib")


In [ ]:
!pip install -U transformers

In [ ]:
import transformers
print(transformers.__version__)
from transformers import TrainingArguments
help(TrainingArguments)

In [ ]:
# Load df (if not already in memory)
import pandas as pd
df = pd.read_csv("scam_dataset_preprocessed.csv")

In [ ]:
# PART B: mBERT fine-tuning with weighted loss
import numpy as np
import pandas as pd
import torch
from datasets import Dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer
import os

# 0) Check GPU
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)
if device == "cpu":
    print("WARNING: No GPU available. Training will be significantly slower.")


# 1) Prepare text and labels (use same train/val/test split indices as SVM)

texts = df['processed_text'].astype(str).tolist()
labels = df['label'].astype(int).values

from sklearn.model_selection import train_test_split
X_temp_text, X_test_text, y_temp_text, y_test_text = train_test_split(texts, labels, test_size=0.15, stratify=labels, random_state=42)
X_train_text, X_val_text, y_train_text, y_val_text = train_test_split(X_temp_text, y_temp_text, test_size=0.17647, stratify=y_temp_text, random_state=42)

train_dataset = Dataset.from_dict({"text": X_train_text, "label": y_train_text})
val_dataset   = Dataset.from_dict({"text": X_val_text, "label": y_val_text})
test_dataset  = Dataset.from_dict({"text": X_test_text, "label": y_test_text})

# 2) Tokenizer & Tokenize
model_name = "bert-base-multilingual-cased"
tokenizer = AutoTokenizer.from_pretrained(model_name)

def preprocess_fn(examples):
    return tokenizer(examples["text"], padding="max_length", truncation=True, max_length=128)

train_dataset = train_dataset.map(preprocess_fn, batched=True)
val_dataset = val_dataset.map(preprocess_fn, batched=True)
test_dataset = test_dataset.map(preprocess_fn, batched=True)

# Set format for PyTorch
train_dataset.set_format(type="torch", columns=["input_ids", "attention_mask", "label"])
val_dataset.set_format(type="torch", columns=["input_ids", "attention_mask", "label"])
test_dataset.set_format(type="torch", columns=["input_ids", "attention_mask", "label"])

# 3) Compute class weights to handle imbalance
import numpy as np
from sklearn.utils.class_weight import compute_class_weight
classes = np.unique(y_train_text)
class_weights = compute_class_weight(class_weight='balanced', classes=classes, y=y_train_text)
# Map to tensor in order of labels 0..N
weight_tensor = torch.tensor(class_weights, dtype=torch.float).to(device)
print("Class weights:", class_weights)

# 4) Load model and modify Trainer to use weighted loss
model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=2).to(device)

# Custom Trainer to apply class weights
from transformers import Trainer

class WeightedTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, num_items_in_batch=None):

        labels = inputs.get("labels")
        outputs = model(input_ids=inputs.get("input_ids"),
                        attention_mask=inputs.get("attention_mask"),
                        labels=None)
        logits = outputs.logits
        loss_fct = torch.nn.CrossEntropyLoss(weight=weight_tensor)
        loss = loss_fct(logits.view(-1, model.config.num_labels), labels.view(-1))
        return (loss, outputs) if return_outputs else loss

# 5) TrainingArguments
training_args = TrainingArguments(
    output_dir="./mbert_scam",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    num_train_epochs=3,
    weight_decay=0.01,
    logging_steps=50,
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    save_total_limit=2,
    fp16=torch.cuda.is_available()  # use mixed precision if GPU available
)

# 6) Metrics function
import evaluate
metric_accuracy = evaluate.load("accuracy")
metric_f1 = evaluate.load("f1")
metric_precision = evaluate.load("precision")
metric_recall = evaluate.load("recall")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    acc = metric_accuracy.compute(predictions=preds, references=labels)["accuracy"]
    f1 = metric_f1.compute(predictions=preds, references=labels, average="binary")["f1"]
    pr = metric_precision.compute(predictions=preds, references=labels, average="binary")["precision"]
    rc = metric_recall.compute(predictions=preds, references=labels, average="binary")["recall"]
    return {"accuracy": acc, "f1": f1, "precision": pr, "recall": rc}

# 7) Trainer and train
trainer = WeightedTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    compute_metrics=compute_metrics,
    tokenizer=tokenizer # Keep tokenizer argument for backward compatibility if needed, though processing_class is preferred
)

trainer.train()

# 8) Evaluate on test set
metrics = trainer.evaluate(test_dataset)
print("mBERT Test metrics:", metrics)

# 9) Save model & tokenizer
trainer.save_model("mbert_scam_model")
tokenizer.save_pretrained("mbert_scam_tokenizer")
print("Saved mBERT model and tokenizer.")

In [ ]:
# Model Evaluation (without ROC-AUC)
from sklearn.metrics import precision_recall_fscore_support, accuracy_score, confusion_matrix, ConfusionMatrixDisplay, precision_recall_curve, auc
import matplotlib.pyplot as plt
import joblib # Import joblib to load the saved model and vectorizer
import pandas as pd # Import pandas to load the dataframe again if needed
from sklearn.feature_extraction.text import TfidfVectorizer # Import TfidfVectorizer

# Load the saved SVM model and TF-IDF vectorizer
try:
    best_svm = joblib.load("svm_tfidf_model.joblib")
    tfidf = joblib.load("tfidf_vectorizer.joblib")
except FileNotFoundError:
    print("Error: svm_tfidf_model.joblib or tfidf_vectorizer.joblib not found.")
    print("Please run the SVM + TF-IDF baseline code first (Part A).")
    exit()

# Load the dataset and prepare data for SVM if not already in memory
df = pd.read_csv("scam_dataset_preprocessed.csv")
X_text = df['final_text_for_traditional_ml'].astype(str).tolist()
y = df['label'].astype(int).values

# Prepare test data for SVM (assuming X_text is already loaded or defined)
X_test_tfidf  = tfidf.transform(X_test)

# SVM predictions on test set:
try:
    from sklearn.model_selection import train_test_split
    texts = df['processed_text'].astype(str).tolist()
    labels = df['label'].astype(int).values
    X_temp_text, X_test_text, y_temp_text, y_test_text = train_test_split(texts, labels, test_size=0.15, stratify=labels, random_state=42)

    X_text = df['final_text_for_traditional_ml'].astype(str).tolist()
    y = df['label'].astype(int).values
    X_temp, X_test, y_temp, y_test = train_test_split(X_text, y, test_size=0.15, stratify=y, random_state=42)
    X_train, X_val, y_train, y_val = train_test_split(X_temp, y_temp, test_size=0.17647, stratify=y_temp, random_state=42)

    X_test_tfidf  = tfidf.transform(X_test)

    svm_preds_test = best_svm.predict(X_test_tfidf)
except NameError:
     print("Error: df, texts, or labels not defined. Please ensure the data loading cells are run.")
     exit()

# mBERT preds: use trainer.predict on tokenized test_dataset
try:
    preds_output = trainer.predict(test_dataset)
    mbert_logits = preds_output.predictions
    mbert_preds_test = np.argmax(mbert_logits, axis=-1)
except NameError:
    print("Error: trainer or test_dataset not defined. Please ensure the mBERT training cell is run.")
    exit()

# Metrics function
def print_metrics(y_true, y_pred, model_name):
    acc = accuracy_score(y_true, y_pred)
    p, r, f1, _ = precision_recall_fscore_support(y_true, y_pred, average='binary')
    print(f"{model_name} -> Acc: {acc:.4f}, Prec: {p:.4f}, Rec: {r:.4f}, F1: {f1:.4f}")
    cm = confusion_matrix(y_true, y_pred)
    disp = ConfusionMatrixDisplay(confusion_matrix=cm)
    disp.plot()
    plt.title(f"Confusion Matrix - {model_name}")
    plt.show()

print_metrics(y_test, svm_preds_test, "SVM")
print_metrics(y_test_text, mbert_preds_test, "mBERT") # Note: using y_test_text for mBERT, which comes from a split on 'processed_text'

In [ ]:
from google.colab import files
import shutil

# Zip and download model
shutil.make_archive('mbert_scam_model', 'zip', 'mbert_scam_model')
files.download('mbert_scam_model.zip')

# Zip and download tokenizer
shutil.make_archive('mbert_scam_tokenizer', 'zip', 'mbert_scam_tokenizer')
files.download('mbert_scam_tokenizer.zip')

shutil.make_archive('mbert_scam', 'zip', 'mbert_scam')
files.download('mbert_scam.zip')

shutil.make_archive('wandb', 'zip', 'wandb')
files.download('wandb.zip')

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!ls "/content/drive/MyDrive/Colab Notebooks/mbert_loaded_files/"

In [ ]:
import shutil
import os

# Define source and destination
SOURCE_PATH = "/content/drive/My Drive/Colab Notebooks/mbert_loaded_files/"
TARGET_PATH = "/content/local_mbert_model/"

# Remove the target folder if it already exists
if os.path.exists(TARGET_PATH):
    shutil.rmtree(TARGET_PATH)

# Copy the entire unzipped folder
print(f"Copying files from Drive to {TARGET_PATH}...")
shutil.copytree(SOURCE_PATH, TARGET_PATH)
print("Copy complete.")

# Load from the reliable local path:
from transformers import AutoModelForSequenceClassification, AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(TARGET_PATH) # local_files_only=True is often implied when loading from /content/
loaded_model = AutoModelForSequenceClassification.from_pretrained(TARGET_PATH)

print(f"Successfully loaded model from reliable local content directory.")

In [ ]:
# Load saved mBERT model and tokenizer (for further processing without the need to run the whole train-test from the start again)

import os
import pandas as pd
from transformers import AutoModelForSequenceClassification, AutoTokenizer, TrainingArguments, Trainer
from datasets import Dataset

# --- CRITICAL SYNCHRONIZATION PARAMETERS ---
# Must match the original training code exactly
MAX_LENGTH = 128
MODEL_BASE_NAME = "bert-base-multilingual-cased"
EVAL_BATCH_SIZE = 32

# NOTE: MBERT_MODEL_PATH must point to the UNZIPPED directory containing the saved model files
# (e.g., '/content/drive/MyDrive/Colab Notebooks/mBERT model/mbert_loaded_files/')
MBERT_MODEL_PATH = "/content/drive/MyDrive/Colab Notebooks/mbert_loaded_files/" 
ABSOLUTE_PATH = os.path.abspath(MBERT_MODEL_PATH)

# --- Step 2a: Load Tokenizer and Model ---
# Load the tokenizer from the *saved* path (recommended for exact match, not just base name)
tokenizer = AutoTokenizer.from_pretrained(ABSOLUTE_PATH, local_files_only=True)
# OR use: tokenizer = AutoTokenizer.from_pretrained(MODEL_BASE_NAME) if you only saved the weights

# Load the saved model weights and config from Google Drive
loaded_model = AutoModelForSequenceClassification.from_pretrained(ABSOLUTE_PATH, local_files_only=True)

# --- Step 2b: Recreate the Test Dataset ---

# Re-run the data split (assuming df, X_text_mbert, y_mbert, X_test_text, y_test_text are available)
# **IMPORTANT:** X_test_text must contain the raw text list used for the test set.

df = pd.read_csv("scam_dataset_preprocessed.csv")

texts = df['processed_text'].astype(str).tolist()
labels = df['label'].astype(int).values

from sklearn.model_selection import train_test_split
X_temp_text, X_test_text, y_temp_text, y_test_text = train_test_split(texts, labels, test_size=0.15, stratify=labels, random_state=42)
X_train_text, X_val_text, y_train_text, y_val_text = train_test_split(X_temp_text, y_temp_text, test_size=0.17647, stratify=y_temp_text, random_state=42)

train_dataset = Dataset.from_dict({"text": X_train_text, "label": y_train_text})
val_dataset   = Dataset.from_dict({"text": X_val_text, "label": y_val_text})
test_dataset  = Dataset.from_dict({"text": X_test_text, "label": y_test_text})

# 1. Define the original preprocess function exactly
def original_preprocess_fn(examples):
    return tokenizer(examples["text"], padding="max_length", truncation=True, max_length=MAX_LENGTH)

# 2. Convert the test data (X_test_text is the text list) into a Hugging Face Dataset format
# NOTE: The column name *must* be "text" to match the original tokenization function.
test_df = pd.DataFrame({'text': X_test_text, 'label': y_test_text})
test_hf_dataset = Dataset.from_pandas(test_df)

# 3. Apply the tokenization
test_dataset = test_hf_dataset.map(original_preprocess_fn, batched=True)

# 4. Set format for PyTorch, matching the columns used in the original code
columns_to_keep = ["input_ids", "attention_mask", "label"]
test_dataset = test_dataset.remove_columns(['text'])
test_dataset.set_format(type="torch", columns=columns_to_keep)


# --- Step 2c: Recreate the Trainer object ---
training_args = TrainingArguments(
    output_dir="./reloaded_trainer_output", # Temporary output directory
    per_device_eval_batch_size=EVAL_BATCH_SIZE,
)

trainer = Trainer(
    model=loaded_model,
    args=training_args,
    # No train_dataset or eval_dataset needed here, only test_dataset for predict
    tokenizer=tokenizer
)
print("mBERT model and Trainer object successfully re-created for prediction.")

In [ ]:
# Model Evaluation (with ROC-AUC)
from sklearn.metrics import precision_recall_fscore_support, accuracy_score, confusion_matrix, ConfusionMatrixDisplay, precision_recall_curve, roc_curve, auc, roc_auc_score
import matplotlib.pyplot as plt
import joblib
import pandas as pd
import numpy as np # Import numpy for array operations
from sklearn.feature_extraction.text import TfidfVectorizer

# --- Data and Model Loading ---

# Load the dataset (Necessary for splitting and getting labels)
try:
    df = pd.read_csv("scam_dataset_preprocessed.csv")
except FileNotFoundError:
    print("Error: scam_dataset_preprocessed.csv not found. Please ensure your dataset is available.")
    exit()

# Load the saved SVM model and TF-IDF vectorizer
try:
    best_svm = joblib.load("svm_tfidf_model.joblib")
    tfidf = joblib.load("tfidf_vectorizer.joblib")
except FileNotFoundError:
    print("Error: svm_tfidf_model.joblib or tfidf_vectorizer.joblib not found. Please run the SVM training code first.")
    exit()

# --- Prepare Test Data and Generate Predictions ---

from sklearn.model_selection import train_test_split

# Assuming 'label' 0 is 'Legit' (Negative) and 1 is 'Scam' (Positive) for binary AUC
# Prepare data for SVM consistency (using 'final_text_for_traditional_ml')
X_text_svm = df['final_text_for_traditional_ml'].astype(str).tolist()
y_svm = df['label'].astype(int).values
X_temp_svm, X_test_svm, y_temp_svm, y_test_svm = train_test_split(X_text_svm, y_svm, test_size=0.15, stratify=y_svm, random_state=42)

# Prepare data for mBERT consistency (using 'processed_text')
X_text_mbert = df['processed_text'].astype(str).tolist()
y_mbert = df['label'].astype(int).values
X_temp_mbert, X_test_mbert, y_temp_mbert, y_test_mbert = train_test_split(X_text_mbert, y_mbert, test_size=0.15, stratify=y_mbert, random_state=42)

# 1. SVM predictions on test set:
try:
    X_test_tfidf = tfidf.transform(X_test_svm)
    svm_preds_test = best_svm.predict(X_test_tfidf)
    # Get probability/score for ROC-AUC. Use decision_function for SVM:
    svm_scores_test = best_svm.decision_function(X_test_tfidf)
    # For binary classification with decision_function, the score is often used directly.
    # Use roc_auc_score(y_true, score)

except Exception as e:
     print(f"Error during SVM prediction: {e}")
     exit()


# 2. mBERT predictions on test set:
# Assuming trainer and test_dataset are available from the execution of the mBERT training cell
try:
    preds_output = trainer.predict(test_dataset)
    mbert_logits = preds_output.predictions
    mbert_preds_test = np.argmax(mbert_logits, axis=-1)

    # Get probabilities for ROC-AUC. Use softmax on logits:
    # Use the probability of the positive class (class 1)
    mbert_probs_test = np.exp(mbert_logits) / np.sum(np.exp(mbert_logits), axis=-1, keepdims=True)
    mbert_scores_test = mbert_probs_test[:, 1]

except NameError:
    print("Error: trainer or test_dataset not defined. Please ensure the mBERT training cell is run.")
    exit()

# --- Metrics Function (Updated for AUC) ---

def print_metrics(y_true, y_pred, y_score, model_name):
    # Standard metrics
    acc = accuracy_score(y_true, y_pred)
    # Note: Use 'binary' average for the two-class problem
    p, r, f1, _ = precision_recall_fscore_support(y_true, y_pred, average='binary', pos_label=1)

    # ROC-AUC Calculation
    # roc_auc_score requires y_score (probabilities or decision scores)
    roc_auc = roc_auc_score(y_true, y_score)

    print(f"\n--- {model_name} Results ---")
    print(f"Accuracy: {acc:.4f}")
    print(f"Precision: {p:.4f}")
    print(f"Recall: {r:.4f}")
    print(f"F1-Score: {f1:.4f}")
    print(f"ROC-AUC: {roc_auc:.4f}")
    print("-" * (len(model_name) + 12))

    # Plotting Confusion Matrix
    cm = confusion_matrix(y_true, y_pred)
    disp = ConfusionMatrixDisplay(confusion_matrix=cm)
    disp.plot()
    plt.title(f"Confusion Matrix - {model_name}")
    plt.show()

    # Plotting ROC Curve
    fpr, tpr, thresholds = roc_curve(y_true, y_score)
    plt.figure()
    plt.plot(fpr, tpr, color='darkorange', lw=2, label=f'ROC curve (area = {roc_auc:.2f})')
    plt.plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--')
    plt.xlim([0.0, 1.0])
    plt.ylim([0.0, 1.05])
    plt.xlabel('False Positive Rate')
    plt.ylabel('True Positive Rate')
    plt.title(f'Receiver Operating Characteristic (ROC) - {model_name}')
    plt.legend(loc="lower right")
    plt.show()


# --- Print Final Metrics ---

# SVM Evaluation
print_metrics(y_test_svm, svm_preds_test, svm_scores_test, "SVM")

# mBERT Evaluation
# Note: y_test_mbert is used here as it aligns with the data used for the mBERT prediction.
print_metrics(y_test_mbert, mbert_preds_test, mbert_scores_test, "mBERT")